In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

np.random.seed(42)

In [2]:
BASE_DIR = Path(
    "paper_implementation"
)

PROFILE_DIR = Path(
    "daily_profiles_24h"
)

MODE1_DIR = (
    BASE_DIR /
    "theft_simulation" /
    "mode_1"
)

MODE1_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
selected_df = pd.read_csv(
    BASE_DIR /
    "selected_150_meters" /
    "selected_150_meter_ids.csv"
)

fraud_df = pd.read_csv(
    BASE_DIR /
    "area_assignments" /
    "fraud_consumers.csv"
)

selected_meters = (
    selected_df["Meter"]
    .astype(str)
    .tolist()
)

fraud_meters = set(
    fraud_df["Meter"]
    .astype(str)
    .tolist()
)

print(
    "Selected Consumers:",
    len(selected_meters)
)

print(
    "Fraud Consumers:",
    len(fraud_meters)
)

Selected Consumers: 150
Fraud Consumers: 27


In [4]:
hour_cols = [
    f"HOUR_{i}"
    for i in range(24)
]

In [5]:
total_modified = 0

log_records = []

for meter in selected_meters:

    df = pd.read_csv(
        PROFILE_DIR /
        f"{meter}.csv"
    )

    # Apply theft only to fraud consumers

    if meter in fraud_meters:

        for row_idx in df.index:

            date = df.loc[
                row_idx,
                "DATE"
            ]

            for hour in hour_cols:

                x_it = float(
                    df.loc[
                        row_idx,
                        hour
                    ]
                )

                # Equation (8)
                alpha_t = np.random.uniform(
                    0.1,
                    0.8
                )

                x_prime = (
                    alpha_t
                    * x_it
                )

                # Save log

                log_records.append({

                    "Meter":
                    meter,

                    "Date":
                    date,

                    "Hour":
                    hour,

                    "x_it":
                    x_it,

                    "alpha_t":
                    alpha_t,

                    "x_prime":
                    x_prime
                })

                # Replace value in Mode-1 dataset

                df.loc[
                    row_idx,
                    hour
                ] = x_prime

                total_modified += 1

    # Save consumer file
    # (fraud modified OR unchanged normal)

    df.to_csv(
        MODE1_DIR /
        f"{meter}.csv",
        index=False
    )

# Save verification log

log_df = pd.DataFrame(
    log_records
)

log_df.to_csv(
    BASE_DIR /
    "theft_simulation" /
    "mode_1_generation_log.csv",
    index=False
)

print(
    "Modified readings:",
    total_modified
)

print(
    "Log rows:",
    len(log_df)
)

Modified readings: 20088
Log rows: 20088


In [6]:
print(
    "Files Generated:",
    len(
        list(
            MODE1_DIR.glob("*.csv")
        )
    )
)

log_df = pd.read_csv(
    BASE_DIR /
    "theft_simulation" /
    "mode_1_generation_log.csv"
)

print(
    "Log Rows:",
    len(log_df)
)

Files Generated: 150
Log Rows: 20088


In [7]:
meter = str(
    fraud_df.iloc[0]["Meter"]
)

original = pd.read_csv(
    PROFILE_DIR /
    f"{meter}.csv"
)

mode1 = pd.read_csv(
    MODE1_DIR /
    f"{meter}.csv"
)

print("Original")
display(original.head(2))

print("Mode 1")
display(mode1.head(2))

Original


,DATE,HOUR_0,HOUR_1,HOUR_2,HOUR_3,HOUR_4,HOUR_5,HOUR_6,HOUR_7,HOUR_8,...,HOUR_14,HOUR_15,HOUR_16,HOUR_17,HOUR_18,HOUR_19,HOUR_20,HOUR_21,HOUR_22,HOUR_23
0,2018-07-01,0.6792,0.6378,0.5856,0.6234,0.5862,0.6450,0.573,0.5886,0.5238,...,0.5352,0.5874,0.5106,0.6054,0.5832,0.6090,0.678,0.6726,0.6552,0.5820
1,2018-07-02,0.6906,0.5634,0.6168,0.5382,0.5832,0.5862,0.579,0.6024,0.5748,...,0.5898,0.5178,0.5850,0.5742,0.5208,0.6024,0.741,0.6432,0.7302,0.6348


Mode 1


,DATE,HOUR_0,HOUR_1,HOUR_2,HOUR_3,HOUR_4,HOUR_5,HOUR_6,HOUR_7,HOUR_8,...,HOUR_14,HOUR_15,HOUR_16,HOUR_17,HOUR_18,HOUR_19,HOUR_20,HOUR_21,HOUR_22,HOUR_23
0,2018-07-01,0.389059,0.240579,0.113083,0.131610,0.448145,0.461640,0.421115,0.467391,0.128761,...,0.275974,0.295413,0.108768,0.260454,0.284269,0.304013,0.167517,0.416736,0.077077,0.202758
1,2018-07-02,0.446230,0.278728,0.174439,0.315637,0.289952,0.385351,0.355806,0.317510,0.253903,...,0.177960,0.054123,0.261710,0.172225,0.319889,0.070338,0.377450,0.488209,0.162168,0.273062


In [8]:
log_df.head(20)

,Meter,Date,Hour,x_it,alpha_t,x_prime
0,6270,2018-07-01,HOUR_0,1.2354,0.362178,0.447435
1,6270,2018-07-01,HOUR_1,1.1880,0.765500,0.909414
2,6270,2018-07-01,HOUR_2,0.8430,0.612396,0.516250
3,6270,2018-07-01,HOUR_3,0.5580,0.519061,0.289636
4,6270,2018-07-01,HOUR_4,0.5826,0.209213,0.121888
5,6270,2018-07-01,HOUR_5,0.5688,0.209196,0.118991
6,6270,2018-07-01,HOUR_6,0.5652,0.140659,0.079500
7,6270,2018-07-01,HOUR_7,0.5376,0.706323,0.379719
8,6270,2018-07-01,HOUR_8,0.8574,0.520781,0.446517
9,6270,2018-07-01,HOUR_9,0.6018,0.595651,0.358463


In [9]:
# VERIFICATION

In [14]:
import numpy as np

unchanged_count = 0

for meter in normal_meters:

    original = pd.read_csv(
        PROFILE_DIR / f"{meter}.csv"
    )

    mode1 = pd.read_csv(
        MODE1_DIR / f"{meter}.csv"
    )

    same = np.allclose(
        original[hour_cols].values,
        mode1[hour_cols].values
    )

    if same:
        unchanged_count += 1

print(
    "Unchanged Normal Consumers:",
    unchanged_count,
    "/",
    len(normal_meters)
)

Unchanged Normal Consumers: 123 / 123


In [15]:
import pandas as pd
from pathlib import Path

total_missing = 0

for file in MODE1_DIR.glob("*.csv"):

    df = pd.read_csv(file)

    total_missing += df.isna().sum().sum()

print(
    "Total Missing Values:",
    total_missing
)

Total Missing Values: 0
